## Part 3: Bragg Peak Masking

Because different corrections are needed when integrating Bragg peaks and diffuse data, it is important to ensure that the signals are separated before processing. It is important to note that a Bragg peaks' extent in $\phi$ can vary depending on the distance from the spindle axis in reciprocal space. And, Bragg peaks may not be perfectly symmetric Gaussian peaks. Peak width depends on crystal mosaicity, and this can even vary during a scan. In addition, the diffraction geometry may not be modeled exactly right, leading to further errors in peak position. A masking function must account for all of these effects. 

Designing a the perfect Bragg peak mask is a tricky problem, and different researchers have approached it various ways. With _mdx2_ we take a simple empirical approach. Here is the algorithm:

- List all of the pixels in the image stack above a certain count threshold ("hot pixels").
- Index all of the hot pixels, converting their position in image space to Miller index (_h_, _k_, _l_).
- Subtract the nearest integer from each Miller index leaving only the fractional part (_Δh_, _Δk_, _Δl_) which varies from -0.5 to 0.5. Because peaks have finite extent, the values of _Δh_, _Δk_, _Δl_ will not all be zero, but will look like a cloud of points.
- Fit an anisotropic 3D Gaussian distribution to the cloud of points.
- Flag outliers beyond a certain sigma threshold ("bad pixels")
- Create a mask for all pixels in the image stack excluding outliers and pixels falling within the 3D Gaussian region according to a sigma threshold.

To execute the masking altorithm in _mdx2_, first perform the strong pixel search:

In [1]:
!mdx2.find_peaks geometry.nxs data.nxs --count_threshold 20

14:28:33 INFO    | Starting mdx2.find_peaks at 2025-12-12 14:28:33
14:28:33 INFO    | Parameters(geom='geometry.nxs', data='data.nxs', count_threshold=20.0, sigma_cutoff=3.0, outfile='peaks.nxs', nproc=1)
14:28:33 INFO    | Loading geometry and image data...


14:28:33 INFO    | Finding pixels above threshold: 20.0
14:28:33 INFO    | Searching for peaks in 600 image chunks (requested n_jobs: 1)...
14:28:33 INFO    | Using backend: SequentialBackend, n_jobs: 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s


[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    0.3s


[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    0.6s


...

[Parallel(n_jobs=1)]: Done 544 tasks      | elapsed:  1.0min


[Parallel(n_jobs=1)]: Done 577 tasks      | elapsed:  1.1min


[Parallel(n_jobs=1)]: Done 600 out of 600 | elapsed:  1.1min finished
14:29:39 INFO    | Peak search completed
14:29:39 INFO    | Found 294235 peak pixels
14:29:39 INFO    | Indexing peaks...


14:29:39 INFO    | Fitting Gaussian peak model...
14:29:39 INFO    | Rejected 6605 outliers (sigma cutoff: 3.0)
14:29:39 INFO    | Peak model r0: [0.033631 0.002241 0.006926]
14:29:39 INFO    | Error ellipsoid semi-axis lengths: [0.039093 0.033736 0.023157]
14:29:39 INFO    | Error ellipsoid principal axis 1: [-0.651304  0.114611 -0.750111]
14:29:39 INFO    | Error ellipsoid principal axis 2: [-0.756288 -0.178675  0.629367]
14:29:39 INFO    | Error ellipsoid principal axis 3: [ 0.061894 -0.97721  -0.203051]
14:29:39 INFO    | Saving peaks to peaks.nxs...


14:29:40 INFO    | Peak finding completed successfully
14:29:40 SUCCESS | mdx2.find_peaks completed in 67.03 seconds


A threshold of 20 was chosen to be 10 times the background rate of ~ 2 photons / pixel, which works well in this case. When the search finishes, a Gaussian is automatically fit to the point cloud and the results are saved to a file (`peaks.nxs`) and printed.

Note that the peak center (`r0`) is not exactly zero, which may indicate a slight error in the diffraction geometry. This error, as well as the extent of the point cloud (`sigma`), should be taken into account when designing a reciprocal space grid for integration. The sampling should not be finer than the geometric errors allow. Here, the Bragg peaks are well localized, and sampling as fine as 20 x 20 x 20 voxels per Bragg peak could be justified (0.5/0.025 = 20). Note that such fine maps consume a lot of computer memory, and thus for the purposes of this tutorial, a coarser map will be used.

Next, create the mask:

In [2]:
!mdx2.mask_peaks geometry.nxs data.nxs peaks.nxs --sigma_cutoff 3

14:29:41 INFO    | Starting mdx2.mask_peaks at 2025-12-12 14:29:41
14:29:41 INFO    | Parameters(geom='geometry.nxs', data='data.nxs', peaks='peaks.nxs', sigma_cutoff=3.0, outfile='mask.nxs', nproc=1, bragg=False)
14:29:41 INFO    | Loading geometry, image data, and peak model...


14:29:41 INFO    | Computing peak mask for 600 image chunks (requested n_jobs: 1)...
14:29:41 INFO    | Using backend: SequentialBackend, n_jobs: 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.8s


[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    3.0s


[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    5.1s


...

[Parallel(n_jobs=1)]: Done 600 out of 600 | elapsed: 12.9min finished
14:42:37 INFO    | Mask computation completed


14:42:38 INFO    | Adding count threshold peaks to mask


14:42:39 INFO    | Masked pixels: 0.17%
14:42:39 INFO    | Saving mask to mask.nxs...


14:42:56 INFO    | Mask creation completed successfully


14:42:57 SUCCESS | mdx2.mask_peaks completed in 795.93 seconds


The `sigma_cutoff` parameter controlls the size of the masked region around the Bragg peak. A value of 3 means that pixels are masked within 3 standard deviations of the peak center according to the model fit above. The function also masks outliers flagged previously. The resulting masks are saved as an image stack in `mask.nxs` and should be inspected using _NeXpy_ before proceeding.